# Importing Library

In [6]:
# ignore all warnings
import warnings
warnings.filterwarnings('ignore')

In [7]:
pip install -U datasets

In [8]:
%%bash
pip install torch_optimizer numpy torch transformers evaluate python-whois --quiet
# The --quiet option is used to suppress the output of a pip command

In [9]:
from datasets import load_dataset
from datasets import Dataset
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Import label encoder
from sklearn import preprocessing, metrics

import itertools
from sklearn.metrics import classification_report, mean_squared_error,confusion_matrix, f1_score, accuracy_score, precision_score, recall_score, auc,roc_curve
from sklearn.model_selection import train_test_split
import random
import math
from collections import Counter
import xgboost as xgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import os
import socket
import whois
from datetime import datetime
import time
from bs4 import BeautifulSoup
import urllib
import bs4
import os

# Data Understanding

In [10]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/El-7areth/url/data/malicious_phish.csv')

### Dataset Information

In [ ]:
print(df.shape)
print(df.info())

In [ ]:
df.groupby('type').apply(lambda x: x.sample(1)).reset_index(drop=True)

In [ ]:
df.isna().sum()

### Data Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

types = df['type'].values

counter_types = Counter(types)

type_names = list(counter_types.keys())
type_values = list(counter_types.values())

sorted_indices = np.argsort(type_values)[::-1]
type_names = [type_names[i] for i in sorted_indices]
type_values = [type_values[i] for i in sorted_indices]

total_count = sum(type_values)
percentages = [value / total_count * 100 for value in type_values]

pattern = '//'

y_pos = np.arange(len(type_names))
plt.figure(1, figsize=(10, 5))
bars = plt.bar(y_pos, type_values, align='center', alpha=0.7, color='none', edgecolor='black', hatch=pattern)

for bar, value, percentage in zip(bars, type_values, percentages):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, f'{value}', ha='center', va='bottom')
    plt.text(bar.get_x() + bar.get_width() / 2, height / 2, f'{percentage:.1f}%', ha='center', va='center',
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'), fontweight='bold')

plt.xticks(y_pos, type_names)
plt.ylabel('Number of URLs')
plt.title('Distribution of URLs per Type')
plt.show()

print(counter_types)

In [ ]:
df['url_len'] = [len(url) for url in df.url]
df.head()

In [ ]:
# Plot distribution of 'url_len' for each 'type'
sns.displot(df, x='url_len', hue='type', kind='kde', fill=True)

# Add labels and title
plt.xlabel('URL Length')
plt.ylabel('Density')
plt.title('Distribution of URL Length by Type')
plt.show()

# Data Preparation

## Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le= LabelEncoder()
le.fit(df["type"])

df["type_code"] = le.transform(df["type"])
df

In [ ]:
le_label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
le_label_mapping

### Splitting
We need to make the data into huggingface dataset pyarrow format, since we gonna upload the dataset into huggingface hub

In [ ]:
df = df[['url', 'type', 'type_code']]
dataset = Dataset.from_pandas(df, preserve_index=False)
dataset

In [ ]:
# split train to 80% of total and test to 20% of total
train_test_dataset = dataset.train_test_split(test_size=0.2, seed=42, shuffle=True)
train_test_dataset

In [ ]:
# split the validation test to 10% of total and test set to 10% of total
val_test_dataset = train_test_dataset['test'].train_test_split(test_size=0.5, seed=42, shuffle=True)
val_test_dataset

In [ ]:
from datasets import DatasetDict

# 80% train, 10% validation, 10% test
dataset = DatasetDict({
    'train': train_test_dataset['train'],
    'val': val_test_dataset['train'],
    'test': val_test_dataset['test'],
})
dataset

### Importing Huggingface Data

In [ ]:
dataset = load_dataset("bgspaditya/byt-mal-minpro")
dataset = dataset.rename_column("type_code", "labels")
dataset

In [ ]:
dataset = load_dataset("bgspaditya/byt-mal-minpro")
dataset = dataset.rename_column("type_code", "labels")
dataset

# Feature Engineering

In [ ]:
df_train = pd.DataFrame(dataset['train'])
df_test = pd.DataFrame(dataset['test'])
df_val = pd.DataFrame(dataset['val'])

## Check IP
To disguise the identity of a website, online criminals frequently use an Internet protocol address instead of the domain name server. This feature will determine if the URL contains an IP address or not

In [ ]:
import re
def having_ip_address(url):
    match = re.search(
        '(([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\.'
        '([01]?\\d\\d?|2[0-4]\\d|25[0-5])\\/)|'  # IPv4
        '((0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\.(0x[0-9a-fA-F]{1,2})\\/)' # IPv4 in hexadecimal
        '(?:[a-fA-F0-9]{1,4}:){7}[a-fA-F0-9]{1,4}', url)  # Ipv6
    if match:
        # print match.group()
        return 1
    else:
        # print 'No matching pattern found'
        return 0
df_train['use_of_ip'] = df_train['url'].apply(lambda i: having_ip_address(i))
df_test['use_of_ip'] = df_test['url'].apply(lambda i: having_ip_address(i))

df_train

## Abnormal URL
The WHOIS database may be used to extract this feature. Identity is commonly
included in the URL of a trustworthy website

In [ ]:
from urllib.parse import urlparse

def abnormal_url(url):
    hostname = urlparse(url).hostname
    hostname = str(hostname)
    match = re.search(hostname, url)
    if match:
        # print match.group()
        return 1
    else:
        # print 'No matching pattern found'
        return 0


df_train['abnormal_url'] = df_train['url'].apply(lambda i: abnormal_url(i))
df_test['abnormal_url'] = df_test['url'].apply(lambda i: abnormal_url(i))
df_train

## Count of dot (.)
The URLs of phishing or malware websites frequently contain more than two subdomains. A dot separates each domain (.). Every URL with more than three dot characters (.)
raises the risk of a malicious website

In [ ]:
df_train['count.'] = df_train['url'].apply(lambda i: i.count('.'))
df_test['count.'] = df_test['url'].apply(lambda i: i.count('.'))

df_train.head()

## Count of WWW (www.)
The majority of secure websites typically only contain one www in their URLs. If
the URL has no www or more than one www, this feature aids in the detection of fraudulent
websites.

In [ ]:
df_train['count-www'] = df_train['url'].apply(lambda i: i.count('www'))
df_test['count-www'] = df_test['url'].apply(lambda i: i.count('www'))
df_train

## Count (@)
If the URL contains the "@" sign, everything before it is ignored

In [ ]:
df_train['count@'] = df_train['url'].apply(lambda i: i.count('@'))
df_test['count@'] = df_test['url'].apply(lambda i: i.count('@'))
df_train

## Count Dir / URL Depth
Websites that have several directories in their URLs are typically suspect

In [ ]:
from urllib.parse import urlparse
def no_of_dir(url):
    urldir = urlparse(url).path
    return urldir.count('/')

df_train['count_dir'] = df_train['url'].apply(lambda i: no_of_dir(i))
df_test['count_dir'] = df_test['url'].apply(lambda i: no_of_dir(i))
df_train

## Count Embed Domain
Knowing how many embedded domains there are will help you spot
dangerous URLs. You may accomplish it by looking for the character "//" in the URL

In [ ]:
def no_of_embed(url):
    urldir = urlparse(url).path
    return urldir.count('//')

df_train['count_embed_domian'] = df_train['url'].apply(lambda i: no_of_embed(i))
df_test['count_embed_domian'] = df_test['url'].apply(lambda i: no_of_embed(i))
df_train

## Count Suspicious URL


In [ ]:
def suspicious_words(url):
    match = re.search('PayPal|login|signin|bank|account|update|free|lucky|service|bonus|ebayisapi|webscr',
                      url)
    if match:
        return 1
    else:
        return 0

df_train['sus_url'] = df_train['url'].apply(lambda i: suspicious_words(i))
df_test['sus_url'] = df_test['url'].apply(lambda i: suspicious_words(i))
df_train

## Count Shortening URL
This feature tells you whether a URL has been shortened using a service, such as bit.ly,
goo.gl, go2l.ink, etc

In [ ]:
def shortening_service(url):
    match = re.search('bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|'
                      'yfrog\.com|migre\.me|ff\.im|tiny\.cc|url4\.eu|twit\.ac|su\.pr|twurl\.nl|snipurl\.com|'
                      'short\.to|BudURL\.com|ping\.fm|post\.ly|Just\.as|bkite\.com|snipr\.com|fic\.kr|loopt\.us|'
                      'doiop\.com|short\.ie|kl\.am|wp\.me|rubyurl\.com|om\.ly|to\.ly|bit\.do|t\.co|lnkd\.in|'
                      'db\.tt|qr\.ae|adf\.ly|goo\.gl|bitly\.com|cur\.lv|tinyurl\.com|ow\.ly|bit\.ly|ity\.im|'
                      'q\.gs|is\.gd|po\.st|bc\.vc|twitthis\.com|u\.to|j\.mp|buzurl\.com|cutt\.us|u\.bb|yourls\.org|'
                      'x\.co|prettylinkpro\.com|scrnch\.me|filoops\.info|vzturl\.com|qr\.net|1url\.com|tweez\.me|v\.gd|'
                      'tr\.im|link\.zip\.net',
                      url)
    if match:
        return 1
    else:
        return 0

df_train['short_url'] = df_train['url'].apply(lambda i: shortening_service(i))
df_test['short_url'] = df_test['url'].apply(lambda i: shortening_service(i))
df_train

## Count HTTPS
Malicious Websites often avoid using HTTPS protocols since they typically demand
user login information and guarantee that online transactions are secure. Hence, whether HTTPS is
present or not is a key component of the URL.

In [ ]:
df_train['count-https'] = df_train['url'].apply(lambda i : i.count('https'))
df_test['count-https'] = df_test['url'].apply(lambda i : i.count('https'))
df_train

## Count HTTP
Safe websites typically have a single HTTP in their URL, but phishing or malicious
websites frequently have several HTTPs.

In [ ]:
df_train['count-http'] = df_train['url'].apply(lambda i : i.count('http'))
df_test['count-http'] = df_test['url'].apply(lambda i : i.count('http'))
df_train

## Count (%)
As we all know, spaces are not permitted in URLs. Normal URL encoding substitutes
the symbol (%) for spaces. Secure websites typically have less spaces in their URLs than
dangerous ones, which means that they have more spaces overall.

In [ ]:
df_train['count%'] = df_train['url'].apply(lambda i: i.count('%'))
df_test['count%'] = df_test['url'].apply(lambda i: i.count('%'))
df_train

## Count (-)
To make a Website appear legitimate, phishers and other cybercriminals frequently add
dashes (-) to the brand name's prefix or suffix. An illustration. www.flipkart-india.com.

In [ ]:
df_train['count-'] = df_train['url'].apply(lambda i: i.count('-'))
df_test['count-'] = df_test['url'].apply(lambda i: i.count('-'))
df_train

## Count (=)
The equal sign (=) in the URL denotes that variables are being sent from one form page
to another. As anybody may alter the values in a URL to change the page, it is regarded as being
riskier

In [ ]:
df_train['count='] = df_train['url'].apply(lambda i: i.count('='))
df_test['count='] = df_test['url'].apply(lambda i: i.count('='))
df_train

## URL Length
To conceal the domain name, attackers frequently utilize lengthy URLs. We
discovered that a safe URL typically has a length of 74 characters

In [ ]:
df_train['url_length'] = df_train['url'].apply(lambda i: len(str(i)))
df_test['url_length'] = df_test['url'].apply(lambda i: len(str(i)))
df_train

## Hostname Length
The hostname's length is a crucial element in identifying fraudulent URLs

In [ ]:
df_train['hostname_length'] = df_train['url'].apply(lambda i: len(urlparse(i).netloc))
df_test['hostname_length'] = df_test['url'].apply(lambda i: len(urlparse(i).netloc))
df_train

In [ ]:
!pip install tld

## First Directory Length
With this feature, you may figure out how long the URL's first directory
is. In order to get the first directory length of the URL, check for the initial '/' and count the length
of the URL up to this point. Installing the Python TLD library is required to obtain directory-level
information. You may install TLD by visiting this page

In [ ]:
#Importing dependencies
from urllib.parse import urlparse
from tld import get_tld
import os.path

#First Directory Length
def fd_length(url):
    urlpath= urlparse(url).path
    try:
        return len(urlpath.split('/')[1])
    except:
        return 0

df_train['fd_length'] = df_train['url'].apply(lambda i: fd_length(i))
df_test['fd_length'] = df_test['url'].apply(lambda i: fd_length(i))
df_train

## TLD Length
one of the domains at the top of the Internet's hierarchical domain
name system is a top-level domain (TLD). For instance, the top-level domain is com in the domain
name www.example.com. So, the length of the TLD is crucial for recognizing fraudulent URLs.
since.com is the most common extension for URLs. TLDs encompassing.

In [ ]:
#Length of Top Level Domain
df_train['tld'] = df_train['url'].apply(lambda i: get_tld(i,fail_silently=True))
df_test['tld'] = df_test['url'].apply(lambda i: get_tld(i,fail_silently=True))

def tld_length(tld):
    try:
        return len(tld)
    except:
        return -1

df_train['tld_length'] = df_train['tld'].apply(lambda i: tld_length(i))
df_test['tld_length'] = df_test['tld'].apply(lambda i: tld_length(i))
df_train

## Digit Count
Suspicious URLs are often those that contain numbers. Counting the amount of
digits in a URL is a key characteristic for identifying fraudulent URLs because safe URLs often
do not include digits

In [ ]:
def digit_count(url):
    digits = 0
    for i in url:
        if i.isnumeric():
            digits = digits + 1
    return digits

df_train['count-digits']= df_train['url'].apply(lambda i: digit_count(i))
df_test['count-digits']= df_test['url'].apply(lambda i: digit_count(i))
df_train

## Letter Count
Another important factor in recognizing fraudulent URLs is the number of letters
in the URL. Attackers typically accomplish this by adding more letters and numbers to the URL in
an effort to lengthen it and conceal the domain name.

In [ ]:
def letter_count(url):
    letters = 0
    for i in url:
        if i.isalpha():
            letters = letters + 1
    return letters

df_train['count-letters']= df_train['url'].apply(lambda i: letter_count(i))
df_test['count-letters']= df_test['url'].apply(lambda i: letter_count(i))
df_train

In [ ]:
df_train = df_train.drop("tld",axis=1)
df_test = df_test.drop("tld",axis=1)
df_train

In [ ]:
df_train.head()

# Data Splitting

In [ ]:
#Predictor Variables
X_train = df_train[['use_of_ip','abnormal_url', 'count.', 'count-www', 'count@',
       'count_dir', 'count_embed_domian', 'short_url', 'count-https',
       'count-http', 'count%', 'count-', 'count=', 'url_length',
       'hostname_length', 'sus_url', 'fd_length', 'tld_length', 'count-digits',
       'count-letters']]

#Target Variable
y_train = df_train['labels']

In [ ]:
#Predictor Variables
X_test = df_test[['use_of_ip','abnormal_url', 'count.', 'count-www', 'count@',
       'count_dir', 'count_embed_domian', 'short_url', 'count-https',
       'count-http', 'count%', 'count-', 'count=', 'url_length',
       'hostname_length', 'sus_url', 'fd_length', 'tld_length', 'count-digits',
       'count-letters']]

#Target Variable
y_test = df_test['labels']

In [ ]:
X_train.to_csv('./x-train.csv')
y_train.to_csv('./y-train.csv')
X_test.to_csv('./x-test.csv')
y_test.to_csv('./y-test.csv')

# Modeling

In [ ]:
eval_df = pd.DataFrame(columns=['Model', 'Accuracy', 'F1-macro', 'F1-micro', 'F1-weighted'])
eval_df

## XGBoost

In [ ]:
xgb = xgb.XGBClassifier(n_estimators= 100)
xgb.fit(X_train,y_train)
y_predXGB = xgb.predict(X_test)
print(classification_report(y_test,y_predXGB))


score = metrics.accuracy_score(y_test, y_predXGB)
print("accuracy:   %0.3f" % score)

In [ ]:
xgb_acc = accuracy_score(y_test, y_predXGB)
xgb_acc

In [ ]:
xgb_f1_macro = f1_score(y_test, y_predXGB, average='macro')
xgb_f1_macro

In [ ]:
xgb_f1_micro = f1_score(y_test, y_predXGB, average='micro')
xgb_f1_micro

In [ ]:
xgb_f1_w = f1_score(y_test, y_predXGB, average='weighted')
xgb_f1_w

In [ ]:
new_eval = {'Model': 'XGB','Accuracy': xgb_acc, 'F1-macro': xgb_f1_macro, 'F1-micro': xgb_f1_micro, 'F1-weighted': xgb_f1_w }
eval_df.loc[len(eval_df)] = new_eval

In [ ]:
CM=confusion_matrix(y_test,y_predXGB,labels=[0,1,2,3])

print(CM)

In [ ]:
xgb_feature = xgb.feature_importances_
xgb_features = xgb_feature.tolist()

In [ ]:
import pickle
# saving model
xgb_pkl = "xgb.pkl"
with open(xgb_pkl, 'wb') as file:
    pickle.dump(xgb, file)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)
y_predRF = rf.predict(X_test)
print(classification_report(y_test,y_predRF))

score = metrics.accuracy_score(y_test, y_predRF)
print("accuracy:   %0.3f" % score)

In [ ]:
rf_acc = accuracy_score(y_test, y_predRF)
rf_acc

In [ ]:
rf_f1_macro = f1_score(y_test, y_predRF, average='macro')
rf_f1_macro

In [ ]:
rf_f1_micro = f1_score(y_test, y_predRF, average='micro')
rf_f1_micro

In [ ]:
rf_f1_w = f1_score(y_test, y_predRF, average='weighted')
rf_f1_w

In [ ]:
new_eval = {'Model': 'RF','Accuracy': rf_acc, 'F1-macro': rf_f1_macro, 'F1-micro': rf_f1_micro, 'F1-weighted': rf_f1_w }
eval_df.loc[len(eval_df)] = new_eval

In [ ]:
CM=confusion_matrix(y_test,y_predRF,labels=[0,1,2,3])

print(CM)

In [ ]:
rf_feature = rf.feature_importances_
rf_features = rf_feature.tolist()

In [ ]:
import pickle
# saving model
rf_pkl = "rf.pkl"
with open(rf_pkl, 'wb') as file:
    pickle.dump(rf, file)

In [ ]:
import os

output_dir = '/content/drive/MyDrive/El-7areth/url/test and train'
os.makedirs(output_dir, exist_ok=True)
print(f"Directory '{output_dir}' created or already exists.")

In [ ]:
import os

output_dir = '/content/drive/MyDrive/El-7areth/url/test and train'

X_train.to_csv(os.path.join(output_dir, 'x-train.csv'), index=False)
y_train.to_csv(os.path.join(output_dir, 'y-train.csv'), index=False)
X_test.to_csv(os.path.join(output_dir, 'x-test.csv'), index=False)
y_test.to_csv(os.path.join(output_dir, 'y-test.csv'), index=False)

print(f"Files saved to: {output_dir}")